# DoMySubs — Google Colab

סטודיו כתוביות לעברית עם WhisperX.

**חשוב:** Runtime → Change runtime type → **T4 GPU** (או טוב יותר)

הרץ את התאים **לפי הסדר** (Shift+Enter).

In [ ]:
# תא 1 — בדיקה ש-GPU זמין
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✓ GPU זמין: {name} ({vram:.1f} GB VRAM)")
else:
    print("✗ אין GPU! לך ל: Runtime → Change runtime type → T4 GPU")
    print("  בלי GPU התמלול יהיה איטי מאוד.")

In [ ]:
# תא 2 — ffmpeg (בדרך כלל כבר מותקן ב-Colab)
# אזהרות apt על "r2u.stat.illinois.edu" — תקינות, אפשר להתעלם.

import shutil
import subprocess

if shutil.which("ffmpeg"):
    ver = subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0]
    print(f"✓ ffmpeg כבר זמין: {ver}")
else:
    print("מתקין ffmpeg...")
    !apt-get install -y -qq ffmpeg 2>/dev/null
    ver = subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0]
    print(f"✓ ffmpeg הותקן: {ver}")

print("\n→ המשך לתא 3 (העלאת ZIP)")

In [ ]:
# תא 3 — העלאת הפרויקט (ZIP מהמחשב)
# במחשב: לחץ ימני על תיקיית DoMySubs → שלח ל → תיקייה דחוסה (zip)
# ואז הרץ תא זה ובחר את הקובץ DoMySubs.zip

from google.colab import files
import zipfile
import os

if not os.path.isdir('/content/DoMySubs/domysubs'):
    print('העלה את קובץ DoMySubs.zip...')
    uploaded = files.upload()  # בחר את ה-ZIP
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print('✓ הפרויקט הועלה')
else:
    print('✓ הפרויקט כבר קיים — מדלג על העלאה')

os.chdir('/content/DoMySubs')
!ls -la

### חלופה מומלצת לתא 3 — שכפול מ-GitHub (עדכונים אוטומטיים)

אחרי שהפרויקט ב-GitHub, הרץ את **תא 3ג** במקום העלאת ZIP.

In [ ]:
# תא 3ב (אופציונלי) — חיבור ל-Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/פרויקטים/DoMySubs')  # עדכן נתיב!
# !ls

In [ ]:
# תא 3ג (מומלץ) — שכפול / עדכון מ-GitHub
# החלף USERNAME בשם המשתמש שלך ב-GitHub

import os

REPO = "https://github.com/YMTzioni/DoMySubs.git"

if os.path.isdir('/content/DoMySubs/.git'):
    os.chdir('/content/DoMySubs')
    !git pull
    print('✓ הפרויקט עודכן מ-GitHub')
else:
    !rm -rf /content/DoMySubs
    !git clone {REPO} /content/DoMySubs
    os.chdir('/content/DoMySubs')
    print('✓ הפרויקט שוכפל מ-GitHub')

!ls -la

In [ ]:
# תא 4 — התקנת חבילות Python (5–15 דקות בפעם הראשונה)
import os
os.chdir('/content/DoMySubs')

%pip install -q whisperx gradio yt-dlp deep-translator python-dotenv
print('✓ חבילות הותקנו')

In [ ]:
# תא 5 — הגדרות אופטימליות ל-GPU ב-Colab
import os

os.environ['DOMYSUBS_DEVICE'] = 'cuda'
os.environ['DOMYSUBS_MODEL'] = 'large-v2'
os.environ['DOMYSUBS_BATCH_SIZE'] = '8'
os.environ['DOMYSUBS_COMPUTE_TYPE'] = 'float16'
os.environ['DOMYSUBS_HOST'] = '0.0.0.0'
os.environ['DOMYSUBS_SHARE'] = 'true'

print('✓ הגדרות GPU הוגדרו')

In [ ]:
# תא 6 — הפעלת הסטודיו
# יופיע קישור public.gradio.live — זה הקישור שלך!
# אל תסגור את התא — הוא מריץ את השרת.

import os
os.chdir('/content/DoMySubs')

# אם קיבלת שגיאת theme — ודא שהעלית ZIP מעודכן עם domysubs/gradio_compat.py
from colab_launch import launch_cloud
launch_cloud()

---
## פתרון בעיות

| בעיה | פתרון |
|------|--------|
| אזהרת apt על `r2u.stat.illinois.edu` | **תקין** — ffmpeg עובד, המשך לתא הבא |
| אין GPU | Runtime → Change runtime type → T4 GPU → Save |
| הקישור נעלם | הרץ שוב תא 6 |
| Colab התנתק | סשן נגמר אחרי ~90 דק (חינמי) — הרץ מחדש |
| שגיאת זיכרון | בתא 5 שנה `DOMYSUBS_MODEL` ל-`medium` ו-`BATCH_SIZE` ל-`4` |
| יוטיוב נכשל | העלה קובץ וידאו ישירות בטאב «וידאו / אודיו» |

**טיפ:** בפעם הראשונה WhisperX מוריד מודלים (~3GB) — זה נורמלי.

In [ ]:
# תא 6ב — תיקון שגיאת theme (הרץ רק אם תא 6 נכשל)
# מעדכן את הקבצים ב-Colab ומרענן את המודולים

import os
os.chdir('/content/DoMySubs')

# gradio_compat.py
open('domysubs/gradio_compat.py', 'w', encoding='utf-8').write('''
from __future__ import annotations
import gradio as gr
from domysubs.ui_theme import CUSTOM_CSS, studio_theme

def blocks_kwargs():
    base = {"title": "DoMySubs Studio"}
    try:
        base["theme"] = studio_theme()
        base["css"] = CUSTOM_CSS
    except Exception:
        pass
    return base

def launch_app(app, server_name="0.0.0.0", server_port=7860, share=False):
    app.launch(server_name=server_name, server_port=server_port, share=share)
''')

# colab_launch.py
open('colab_launch.py', 'w', encoding='utf-8').write('''
import os
from domysubs.gradio_compat import launch_app
from domysubs.studio import create_app

def launch_cloud(host=None, port=None, share=None):
    host = host or os.getenv("DOMYSUBS_HOST", "0.0.0.0")
    port = int(port or os.getenv("DOMYSUBS_PORT", "7860"))
    if share is None:
        share = os.getenv("DOMYSUBS_SHARE", "true").lower() in ("1", "true", "yes")
    if os.getenv("COLAB_RELEASE_TAG"):
        share = True
        os.environ.setdefault("DOMYSUBS_DEVICE", "cuda")
        os.environ.setdefault("DOMYSUBS_COMPUTE_TYPE", "float16")
    launch_app(create_app(), server_name=host, server_port=port, share=share)
''')

# רענון מודולים (חשוב!)
import sys
for mod in list(sys.modules):
    if mod.startswith('domysubs') or mod == 'colab_launch':
        del sys.modules[mod]

print('✓ תיקון הוחל — הרץ שוב את תא 6')